# EEG Preprocessing simf10

This notebook processes EEG data from the SIMF dataset. The recordings capture 20 simulated spikes.

## 🧪 - Experiment setup
- `eeg`: 'raw' EEG.

In [1]:
%matplotlib widget
import mindmap as mm
import mne
import sys

# reader
dm = mm.io.TutoManager("../../config.yml")
sr = dm.get_subject("simf10")

# data
eeg = sr.load("eeg", task="rest", run="11")

/home/nino/Code/mindmap/mindmap/io/tutorial.py:59: RuntimeWarning: participants.tsv file not found for /home/nino/Code/mindmap/data/sub-simf10/eeg/sub-simf10_task-rest_run-11_eeg.vhdr
  subreader.add_datatype("eeg", "raw", mne_bids.BIDSPath, lambda f: mne_bids.read_raw_bids(f, verbose="warning").load_data(verbose="warning"))


## 🧬 - Data Processing
From Raw to Epochs

In [2]:
# Average referencing
print("\033[95mSet average referencing ...\033[0m")
eeg = eeg.set_eeg_reference(ref_channels="average", projection=True)
sys.stdout.flush()  # prints the MNE warning here rather then at the end of code


# Epoching
print("\n\033[95mSplit data into epochs (this step automatically applies projections and baseline correction!) ...\033[0m")
events = mne.events_from_annotations(eeg)[0]
epochs = mne.Epochs(eeg, events, tmin=-0.45, tmax=0.45, baseline=[-0.44, -0.2])

Set average referencing ...
EEG channel type selected for re-referencing
Adding average EEG reference projection.
1 projection items deactivated
Average reference projection was added, but has not been applied yet. Use the apply_proj method to apply it.

Split data into epochs (this step automatically applies projections and baseline correction!) ...
Used Annotations descriptions: [np.str_('1.0')]
Not setting metadata
20 matching events found
Setting baseline interval to [-0.44, -0.2] s
Applying baseline correction (mode: mean)
Created an SSP operator (subspace dimension = 1)
1 projection items activated


Compute noise covariance matrix

In [3]:
print("\033[95mComputes noise covariance (applies projectors)...\033[0m")
noise_cov = mne.compute_covariance(epochs, tmax=-0.2, method="empirical")

Computes noise covariance (applies projectors)...
Using data from preloaded Raw for 20 events and 225 original time points ...
0 bad epochs dropped
    Created an SSP operator (subspace dimension = 1)
    Setting small EEG eigenvalues to zero (without PCA)
Reducing data rank from 203 -> 202
Estimating covariance using EMPIRICAL
Done.
Number of samples used : 1260
[done]


From epochs to evoked (average signal)

In [4]:
print("\033[95mAveraging epochs...\033[0m")
evoked = epochs.average()

Averaging epochs...


## 📊 - Visualization
There are many aspects of EEG data to visualize. Let’s generate an MNE report so that all plots can be explored in a web browser.

In [5]:
mm.viz.create_pipeline_report(evoked, noise_cov, subject_id = sr.sub)

Embedding : jquery-3.6.0.min.js
Embedding : bootstrap.bundle.min.js
Embedding : bootstrap.min.css
Embedding : bootstrap-table/bootstrap-table.min.js
Embedding : bootstrap-table/bootstrap-table.min.css
Embedding : bootstrap-table/bootstrap-table-copy-rows.min.js
Embedding : bootstrap-table/bootstrap-table-export.min.js
Embedding : bootstrap-table/tableExport.min.js
Embedding : bootstrap-icons/bootstrap-icons.mne.min.css
Embedding : highlightjs/highlight.min.js
Embedding : highlightjs/atom-one-dark-reasonable.min.css
Projections have already been applied. Setting proj attribute to True.


[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    1.7s
[Parallel(n_jobs=1)]: Done  21 out of  21 | elapsed:    2.0s finished


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Computing rank from covariance with rank=None
    Using tolerance 9.6e-13 (2.2e-16 eps * 203 dim * 21  max singular value)
    Estimated rank (eeg): 202
    EEG: rank 202 computed from 203 data channels with 1 projector
    Created an SSP operator (subspace dimension = 1)
Computing rank from covariance with rank={'eeg': 202}
    Setting small EEG eigenvalues to zero (without PCA)
    Created the whitener using a noise covariance matrix with rank 202 (1 small eigenvalues omitted)
Projections have already been applied. Setting proj attribute to True.
Saving report to : /home/nino/Code/mindmap/tutorials/analysis/eeg/Report_simf10.html
Report saved for simf10
